# Import

In [ ]:
import cv2
import os
import json
import shutil
from pathlib import Path

# Setup

In [ ]:
WORKSPACE_DIR = Path("./AI_Challenge")
FINAL_DATASET_DIR = WORKSPACE_DIR / "final_dataset"
KEYFRAMES_DIR = FINAL_DATASET_DIR / "keyframes"
KEYFRAMES_JSONL = FINAL_DATASET_DIR / "keyframes.jsonl"

# Thư mục chứa frame bị loại (Trash bin)
TRASH_DIR = WORKSPACE_DIR / "trash_bin"
TRASH_DUP = TRASH_DIR / "duplicate"

for d in [TRASH_DUP]:
    d.mkdir(parents=True, exist_ok=True)

# File checkpoint ghi nhận tiến độ
CHECKPOINT_LOG = WORKSPACE_DIR / "clean_frames_checkpoint.log"

# Ngưỡng đánh giá (có thể tinh chỉnh)
SIMILARITY_THRESHOLD = 0.95

# Set lưu trữ các frame bị xóa
deleted_frame_ids = set()

# Functions

In [ ]:
def is_duplicate(img1, img2, threshold=SIMILARITY_THRESHOLD):
    """So sánh độ tương đồng Histogram"""
    hist1 = cv2.calcHist([img1], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    hist2 = cv2.calcHist([img2], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist1, hist1)
    cv2.normalize(hist2, hist2)
    return cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL) > threshold

def get_processed_videos():
    """Đọc danh sách video đã xử lý xong từ log"""
    if not CHECKPOINT_LOG.exists(): return set()
    with open(CHECKPOINT_LOG, 'r') as f:
        return set(line.strip() for line in f)

def mark_processed(video_id):
    """Ghi video đã xử lý xong vào log"""
    with open(CHECKPOINT_LOG, 'a') as f:
        f.write(f"{video_id}\n")

# Delete Frame Loop

In [ ]:
processed_videos = get_processed_videos()
print(f"Bắt đầu quét. Bỏ qua {len(processed_videos)} video đã hoàn tất.")

try:
    for video_folder in sorted(KEYFRAMES_DIR.iterdir()):
        if not video_folder.is_dir() or video_folder.name in processed_videos:
            continue
            
        print(f"Đang xử lý: {video_folder.name}...")
        frame_files = sorted(video_folder.glob("*.jpg"))
        prev_image = None
        
        for frame_path in frame_files:
            img = cv2.imread(str(frame_path))
            if img is None: continue
            
            rel_frame_path = f"keyframes/{video_folder.name}/{frame_path.name}"
            trash_filename = f"{video_folder.name}_{frame_path.name}"
            
            # Kiểm tra trùng lặp
            if prev_image is not None and is_duplicate(prev_image, img):
                deleted_frame_ids.add(rel_frame_path)
                shutil.move(str(frame_path), str(TRASH_DUP / trash_filename))
                continue
                
            prev_image = img
            
        # Ghi nhận thư mục đã quét xong 100%
        mark_processed(video_folder.name)

except KeyboardInterrupt:
    print("\n[!] Đã dừng tiến trình thủ công (Manual Stop).")

print(f"\n=> Tổng số frame đã dọn dẹp trong biến lưu trữ hiện tại: {len(deleted_frame_ids)}")

# Update Jsonl

In [ ]:
if not deleted_frame_ids:
    print("Không có frame nào bị xóa. Không cần cập nhật JSONL.")
elif not KEYFRAMES_JSONL.exists():
    print("Không tìm thấy file keyframes.jsonl gốc!")
else:
    print("Đang đồng bộ lại file keyframes.jsonl...")
    temp_jsonl = KEYFRAMES_JSONL.with_suffix('.tmp')
    valid_count = 0
    
    with open(KEYFRAMES_JSONL, 'r', encoding='utf-8') as fin, \
         open(temp_jsonl, 'w', encoding='utf-8') as fout:
        
        for line in fin:
            data = json.loads(line)
            
            # Chỉ loại bỏ những record nằm trong danh sách Duplicate
            if data['keyframe_path'] not in deleted_frame_ids:
                fout.write(json.dumps(data) + '\n')
                valid_count += 1

    os.replace(temp_jsonl, KEYFRAMES_JSONL)
    deleted_frame_ids.clear()
    
    print(f"✅ Đã cập nhật thành công! Giữ lại {valid_count} records hợp lệ.")